# ModernBERT - Run the Pipeline from GitHub

One-shot entry point for the `Compartment` repo (`main.py` + `src/` package).

**How to use**

1. Edit the cell below: set `GITHUB_URL` to your repository (only needed if you are not using a GitHub input) and decide `USE_KAGGLE_INPUT`.
2. Make sure the **Compartment** dataset is attached as an input (it provides `dataset/` and `trial/`). 
3. Click **Run All**.

**What happens**

- locates/clones the repo (GitHub input mount or `git clone`)
- ensures the Python environment is ready
- runs `main.py <MODE>` through the CLI (config + validation happen there)
- prints metrics, previews the submission and creates `submission.zip`


## 1. Locate / clone the repository
Set the two variables at the top of this cell, then run it.


In [ ]:
import glob, json, os, subprocess, sys, zipfile

# ==== 1. Point at your repo ==========================================
USE_KAGGLE_INPUT = True   # True  -> GitHub repo mounted via "Add Input -> GitHub"
                          # False -> git clone the URL below into /kaggle/working
GITHUB_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # <-- EDIT ME

# ==== 2. Locate the repo root ========================================
def sh(cmd, cwd=None):
    print('>>>', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    sys.stdout.write(r.stdout)
    sys.stderr.write(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')

REPO = None
if USE_KAGGLE_INPUT:
    mains = sorted(glob.glob('/kaggle/input/github/**/main.py', recursive=True))
    if mains:
        REPO = os.path.dirname(mains[0])
if REPO is None or not os.path.isdir(os.path.join(REPO, 'src')):
    REPO = '/kaggle/working/compartment'
    if not os.path.isdir(os.path.join(REPO, '.git')):
        sh(f'git clone --depth 1 {GITHUB_URL} {REPO}')

print('REPO =', REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)
print('CWD =', os.getcwd())


## 2. Environment
Only installs on Kaggle if a dependency is missing (they all ship pre-installed).


In [ ]:
import subprocess, sys

# On Kaggle, torch / transformers are preinstalled. Only install when missing.
check = subprocess.run(
    [sys.executable, '-c', 'import torch, transformers, pandas, numpy, sklearn, scipy'],
    capture_output=True,
)
if check.returncode != 0:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch', 'transformers>=4.41', 'pandas', 'numpy', 'scikit-learn', 'scipy'],
        check=True,
    )
    print('Installed missing packages.')
else:
    print('Environment OK (torch, transformers, pandas, numpy, sklearn, scipy).')


## 3. Choose the run
Set `MODE` to `train80`, `train5` or `predict`. Append any CLI override to
`EXTRA_TRAIN` / `EXTRA_PREDICT` (e.g. `' --epochs 8 --batch 16 --ccc-weight 0.5'`).


In [ ]:
# ==== 3. The run ======================================================
MODE = 'train5'          # 'train80' = quick 80/20 split
                         # 'train5'  = stratified 5-fold CV (default)
                         # 'predict' = trial predictions + submission
DO_PREDICT = True        # run "main.py predict" after training (needs checkpoints)
EXTRA_TRAIN = ''         # optional CLI overrides, e.g. ' --epochs 8 --batch 16'
EXTRA_PREDICT = ''       # e.g. ' --predict-mode single --seed 7'

# Data location is auto-detected (Kaggle dataset input or ./dataset) but can be
# pinned with ' --data-path <dir>'. Outputs go to /kaggle/working (or ./output).


In [ ]:
import os, subprocess, sys

PY = sys.executable

if MODE == 'predict':
    subprocess.run(f'{PY} main.py predict {EXTRA_PREDICT}'.split(), check=True, cwd=os.getcwd())
else:
    subprocess.run(f'{PY} main.py {MODE} {EXTRA_TRAIN}'.split(), check=True, cwd=os.getcwd())
    if DO_PREDICT:
        print('\n--- now generating the submission ---')
        subprocess.run(f'{PY} main.py predict {EXTRA_PREDICT}'.split(), check=True, cwd=os.getcwd())


## 4. Results
Printed from the JSON artifacts that `main.py` writes next to the checkpoints.


In [ ]:
import json, os

OUT = '/kaggle/working'
for fname in ('metrics.json', 'trial_metrics.json'):
    path = os.path.join(OUT, fname)
    if os.path.isfile(path):
        print('=' * 20, fname, '=' * 20)
        print(json.dumps(json.load(open(path)), indent=2))


## 5. Submission
Preview the predictions and get a Kaggle-ready `submission.zip` (TSV, no header).


In [ ]:
import os, zipfile

import pandas as pd

sub = os.path.join('/kaggle/working', 'submission', 'en-nn-trial-pred.tsv')
if not os.path.isfile(sub):
    print('No submission yet - run with MODE=predict or DO_PREDICT=True.')
else:
    df = pd.read_csv(sub, sep='\t', header=None, names=['tID', 'Modifier', 'Head'])
    print(df.head(10).to_string(index=False))
    print(f'\nTotal rows: {len(df)}')

    zip_path = os.path.join('/kaggle/working', 'submission', 'submission.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(sub, arcname=os.path.basename(sub))
    print('Kaggle-ready submission:', zip_path)
